In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- locate the Millennium workbook ---
here = Path.cwd()
matches = list(here.rglob("a-millennium-of-macroeconomic-data-for-the-uk.xlsx"))
if not matches:
    for parent in here.parents:
        matches = list(parent.rglob("a-millennium-of-macroeconomic-data-for-the-uk.xlsx"))
        if matches: break
xl = matches[0]
print("Using:", xl)

# --- 1. Primary balance (£mn), tab A28, col 23, 1900-2016 ---
a28 = pd.read_excel(xl, sheet_name="A28. Public Sector Borrowing", header=None)
pb = a28.iloc[8:, [0, 23]].copy()
pb.columns = ["year", "pb_mn"]
pb["year"] = pd.to_numeric(pb["year"], errors="coerce")
pb["pb_mn"] = pd.to_numeric(pb["pb_mn"], errors="coerce")
pb = pb.dropna().query("1900 <= year <= 2016")

# --- 2. Nominal GDP (£mn), tab A9, col 3 ---
a9 = pd.read_excel(xl, sheet_name="A9. Nominal GDP (A)", header=None)
gdp = a9.iloc[5:, [0, 3]].copy()
gdp.columns = ["year", "gdp_mn"]
gdp["year"] = pd.to_numeric(gdp["year"], errors="coerce")
gdp["gdp_mn"] = pd.to_numeric(gdp["gdp_mn"], errors="coerce")
gdp = gdp.dropna()

# --- 3. Debt (£mn), tab A29: splice consolidated (col 31, to 1978) + PSND (col 32, from 1974) ---
a29 = pd.read_excel(xl, sheet_name="A29. The National Debt", header=None)
yr  = a29.iloc[6:, 0].apply(lambda s: int(str(s)[:4]) if str(s)[:4].isdigit() else np.nan)
old = pd.to_numeric(a29.iloc[6:, 31], errors="coerce")   # consolidated national debt
new = pd.to_numeric(a29.iloc[6:, 32], errors="coerce")   # PSND
debt = pd.DataFrame({"year": yr, "old": old, "new": new}).dropna(subset=["year"])
debt["year"] = debt["year"].astype(int)
debt["debt_mn"] = debt["new"].fillna(debt["old"])        # prefer PSND where available
debt = debt.groupby("year", as_index=False)["debt_mn"].mean()

# --- merge and convert to % of GDP ---
df = (pb.merge(gdp, on="year").merge(debt, on="year"))
df["pb"]   = df["pb_mn"]   / df["gdp_mn"] * 100   # primary balance, % GDP
df["debt"] = df["debt_mn"] / df["gdp_mn"] * 100   # debt, % GDP
df = df.sort_values("year").reset_index(drop=True)
df["debt_lag"] = df["debt"].shift(1)
# flag war years (handle separately later)
df["war"] = df["year"].isin(list(range(1914,1919)) + list(range(1939,1946))).astype(int)

# --- SANITY CHECK ---
print(f"\nSample: {int(df.year.min())}–{int(df.year.max())}   n = {len(df)}")
print("\nSummary (should look sensible — debt peaks ~250% post-WWII, pb mostly small negatives):")
print(df[["pb", "debt"]].describe().round(1))
print("\nSplice overlap check (1974-78: old consolidated vs new PSND should be roughly close):")
print(debt_overlap := a29.iloc[6:, [0,31,32]].apply(pd.to_numeric, errors="coerce")
      .rename(columns={0:"y",31:"old",32:"new"}).dropna().query("1974<=y<=1978").to_string(index=False))
print("\nFirst & last rows:")
print(df[["year","pb","debt","debt_lag"]].head(3).to_string(index=False))
print(df[["year","pb","debt","debt_lag"]].tail(3).to_string(index=False))

df.to_csv("frf_millennium.csv", index=False)
print("\nsaved frf_millennium.csv")

Using: /Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Research/Data/a-millennium-of-macroeconomic-data-for-the-uk.xlsx

Sample: 1900–2016   n = 117

Summary (should look sensible — debt peaks ~250% post-WWII, pb mostly small negatives):
          pb   debt
count  117.0  117.0
mean     0.2  100.3
std      7.3   64.9
min    -24.0   22.8
25%     -1.1   38.9
50%      1.9   84.9
75%      4.0  157.3
max      9.8  264.1

Splice overlap check (1974-78: old consolidated vs new PSND should be roughly close):
Empty DataFrame
Columns: [y, old, new]
Index: []

First & last rows:
 year        pb      debt  debt_lag
 1900 -2.079966 37.636715       NaN
 1901 -2.644997 40.648315 37.636715
 1902 -1.264495 42.060268 40.648315
 year        pb       debt   debt_lag
 2014 -2.508066 101.910693 117.197135
 2015 -1.686056 101.983699 101.910693
 2016 -0.299180 105.620949 101.983699

saved frf_millennium.csv


In [2]:
import pandas as pd, numpy as np
a29 = pd.read_excel(xl, sheet_name="A29. The National Debt", header=None)
chk = pd.DataFrame({
    "year": a29.iloc[6:, 0].apply(lambda s: int(str(s)[:4]) if str(s)[:4].isdigit() else np.nan),
    "old_consolidated": pd.to_numeric(a29.iloc[6:, 31], errors="coerce"),
    "new_PSND":         pd.to_numeric(a29.iloc[6:, 32], errors="coerce"),
}).dropna(subset=["year"])
chk["year"] = chk["year"].astype(int)
print(chk[(chk.year >= 1972) & (chk.year <= 1980)].to_string(index=False))

 year  old_consolidated  new_PSND
 1972           36885.0       NaN
 1973           40125.0       NaN
 1974           45925.0   52098.0
 1975           56572.0   64739.0
 1976           67166.0   73575.0
 1977           79180.0   79541.0
 1978           86885.0   88605.0
 1979               NaN   98192.0
 1980               NaN  113835.0


In [3]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
import warnings; warnings.filterwarnings("ignore")

df = pd.read_csv("frf_millennium.csv")

def test(series, name):
    s = series.dropna()
    adf_p = adfuller(s, autolag="AIC")[1]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kpss_p = kpss(s, regression="c", nlags="auto")[1]
    # ADF: low p = stationary;  KPSS: low p = non-stationary
    verdict = ("I(0) stationary" if adf_p < 0.05 and kpss_p > 0.05 else
               "I(1) non-stationary" if adf_p > 0.05 and kpss_p < 0.05 else
               "ambiguous")
    print(f"{name:18s} ADF p={adf_p:.3f}  KPSS p={kpss_p:.3f}  -> {verdict}")

print("LEVELS:")
test(df["pb"], "primary balance")
test(df["debt"], "debt/GDP")
print("\nFIRST DIFFERENCES:")
test(df["pb"].diff(), "Δ primary balance")
test(df["debt"].diff(), "Δ debt/GDP")

LEVELS:
primary balance    ADF p=0.000  KPSS p=0.100  -> I(0) stationary
debt/GDP           ADF p=0.340  KPSS p=0.049  -> I(1) non-stationary

FIRST DIFFERENCES:
Δ primary balance  ADF p=0.000  KPSS p=0.100  -> I(0) stationary
Δ debt/GDP         ADF p=0.000  KPSS p=0.100  -> I(0) stationary


In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv("frf_millennium.csv").sort_values("year").reset_index(drop=True)

# first differences (both I(0) -> balanced regression)
df["d_pb"]   = df["pb"].diff()
df["d_debt"] = df["debt"].diff()
# lag of the debt *change* (does pb change respond to last year's debt change?)
df["d_debt_lag"] = df["d_debt"].shift(1)

reg = df.dropna(subset=["d_pb", "d_debt_lag", "war"]).copy()

def run(data, label):
    m = smf.ols("d_pb ~ d_debt_lag", data=data).fit(cov_type="HAC", cov_kwds={"maxlags":3})
    b, se, p = m.params["d_debt_lag"], m.bse["d_debt_lag"], m.pvalues["d_debt_lag"]
    star = "***" if p<.01 else "**" if p<.05 else "*" if p<.1 else ""
    print(f"{label:22s} n={len(data):3d}  beta={b:+.4f} (se {se:.4f}) p={p:.3f} {star}")

print("DIFFERENCED FISCAL REACTION FUNCTION  (Δpb on lagged Δdebt, HAC SEs)")
print("beta>0 = when debt rose last year, the primary balance improves this year (correcting)")
print("-"*72)
run(reg, "Full 1902-2016")
run(reg[reg.war==0], "Ex-war years")
print("-"*72)
for lo,hi in [(1902,1945),(1946,1979),(1980,1999),(2000,2016)]:
    s = reg[(reg.year>=lo)&(reg.year<=hi)]
    if len(s) > 8:
        run(s, f"{lo}-{hi}")

DIFFERENCED FISCAL REACTION FUNCTION  (Δpb on lagged Δdebt, HAC SEs)
beta>0 = when debt rose last year, the primary balance improves this year (correcting)
------------------------------------------------------------------------
Full 1902-2016         n=115  beta=+0.0065 (se 0.0344) p=0.851 
Ex-war years           n=103  beta=+0.0185 (se 0.0434) p=0.670 
------------------------------------------------------------------------
1902-1945              n= 44  beta=+0.0395 (se 0.0519) p=0.447 
1946-1979              n= 34  beta=+0.1430 (se 0.0777) p=0.066 *
1980-1999              n= 20  beta=+0.0674 (se 0.0681) p=0.323 
2000-2016              n= 17  beta=-0.0370 (se 0.0051) p=0.000 ***


In [5]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv("frf_millennium.csv").sort_values("year").reset_index(drop=True)
df["d_pb"] = df["pb"].diff()
df["d_debt"] = df["debt"].diff()
df["d_debt_lag"] = df["d_debt"].shift(1)
reg = df.dropna(subset=["d_pb","d_debt_lag"]).copy()

def run(data, label):
    m = smf.ols("d_pb ~ d_debt_lag", data=data).fit(cov_type="HAC", cov_kwds={"maxlags":2})
    b,se,p = m.params["d_debt_lag"], m.bse["d_debt_lag"], m.pvalues["d_debt_lag"]
    print(f"{label:28s} n={len(data):3d}  beta={b:+.4f} (se {se:.4f}) p={p:.3f}")

print("2000-2016 robustness:")
run(reg[(reg.year>=2000)&(reg.year<=2016)], "2000-2016 (all)")
run(reg[(reg.year>=2000)&(reg.year<=2016)&(~reg.year.isin([2008,2009]))], "2000-2016 ex-GFC")
run(reg[(reg.year>=2000)&(reg.year<=2016)&(~reg.year.isin([2008,2009,2020]))], "ex-GFC (no COVID anyway)")
# also: does the post-war positive survive ex-shocks?
print("\n1946-1979 robustness:")
run(reg[(reg.year>=1946)&(reg.year<=1979)], "1946-1979 (all)")

2000-2016 robustness:
2000-2016 (all)              n= 17  beta=-0.0370 (se 0.0055) p=0.000
2000-2016 ex-GFC             n= 15  beta=+0.0255 (se 0.0212) p=0.229
ex-GFC (no COVID anyway)     n= 15  beta=+0.0255 (se 0.0212) p=0.229

1946-1979 robustness:
1946-1979 (all)              n= 34  beta=+0.1430 (se 0.0894) p=0.110
